In [1]:
!pip install pyspark pandas openpyxl

In [2]:
import zipfile
import pandas as pd

In [6]:
import zipfile

# Correct file name without the leading slash
zip_filename = 'snack-analysis (1).zip'

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('Extracted_snack_data')

print('Extracted Successfully')

Extracted Successfully


In [7]:
import os
import pandas as pd

# Let's print out what folders and files actually exist inside Extracted_snack_data
print(os.walk('Extracted_snack_data'))
for root, dirs, files in os.walk('Extracted_snack_data'):
    print(f"Path: {root}")
    print(f"Files: {files}\n")

<generator object walk at 0x7b24b7c47400>
Path: Extracted_snack_data
Files: []

Path: Extracted_snack_data/archive
Files: ['menu.xlsx', 'starbucks-menu-nutrition-drinks.xlsx', 'starbucks_drinkMenu_expanded.xlsx', 'starbucks-menu-nutrition-food.xlsx']



In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import pandas as pd

# Define the correct path based on the directory walk output
base_path = 'Extracted_snack_data/archive'

# Convert the files to CSV
pd.read_excel(f'{base_path}/menu.xlsx').to_csv('mcdonalds_menu.csv', index=False)
pd.read_excel(f'{base_path}/starbucks-menu-nutrition-drinks.xlsx').to_csv('starbucks_drinks.csv', index=False)
pd.read_excel(f'{base_path}/starbucks-menu-nutrition-food.xlsx').to_csv('starbucks_food.csv', index=False)
pd.read_excel(f'{base_path}/starbucks_drinkMenu_expanded.xlsx').to_csv('starbucks_drink_expanded.csv', index=False)

print("All files successfully converted to CSV and ready for PySpark!")

All files successfully converted to CSV and ready for PySpark!


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("SnackAnalysisStarbucksVsMcDonalds") \
    .getOrCreate()

# 2. Load the converted CSV files into PySpark DataFrames
mcd_df = spark.read.option("header", "true").option("inferSchema", "true").csv("mcdonalds_menu.csv")
starbucks_drinks_df = spark.read.option("header", "true").option("inferSchema", "true").csv("starbucks_drinks.csv")

# 3. Quick schema check and data preview
print("--- McDonald's Schema ---")
mcd_df.printSchema()

print("--- McDonald's Sample Data ---")
mcd_df.show(3, truncate=False)

--- McDonald's Schema ---
root
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Serving Size: string (nullable = true)
 |-- Calories: integer (nullable = true)
 |-- Calories from Fat: integer (nullable = true)
 |-- Total Fat: double (nullable = true)
 |-- Total Fat (% Daily Value): integer (nullable = true)
 |-- Saturated Fat: double (nullable = true)
 |-- Saturated Fat (% Daily Value): integer (nullable = true)
 |-- Trans Fat: double (nullable = true)
 |-- Cholesterol: integer (nullable = true)
 |-- Cholesterol (% Daily Value): integer (nullable = true)
 |-- Sodium: integer (nullable = true)
 |-- Sodium (% Daily Value): integer (nullable = true)
 |-- Carbohydrates: integer (nullable = true)
 |-- Carbohydrates (% Daily Value): integer (nullable = true)
 |-- Dietary Fiber: integer (nullable = true)
 |-- Dietary Fiber (% Daily Value): integer (nullable = true)
 |-- Sugars: integer (nullable = true)
 |-- Protein: integer (nullable = true)
 |-- Vitamin A (% 

In [13]:
print(starbucks_drinks_df.columns)

['Unnamed: 0', 'Calories', 'Fat (g)', 'Carb. (g)', 'Fiber (g)', 'Protein', 'Sodium']


In [17]:
from pyspark.sql.functions import avg, round

# Run the average aggregation with backticks around special column names
cleaned_df.select(
    round(avg("Calories"), 2).alias("Avg_Calories"),
    round(avg("Fat (g)"), 2).alias("Avg_Fat"),
    round(avg("`Carb. (g)`"), 2).alias("Avg_Carbs"),
    round(avg("`Fiber (g)`"), 2).alias("Avg_Fiber"),
    round(avg("Protein"), 2).alias("Avg_Protein"),
    round(avg("Sodium"), 2).alias("Avg_Sodium")
).show()

+------------+-------+---------+---------+-----------+----------+
|Avg_Calories|Avg_Fat|Avg_Carbs|Avg_Fiber|Avg_Protein|Avg_Sodium|
+------------+-------+---------+---------+-----------+----------+
|      135.16|   2.34|    24.74|     0.45|       4.15|     57.93|
+------------+-------+---------+---------+-----------+----------+

